# Bootstrap Flow Debugger

This notebook walks through the exact failure path behind `marts.v_device_risk_profile` and lets you inspect each dependency one step at a time.

By default, the notebook reproduces the current failure: `marts.mart_device_monthly_behavior` does not exist when the view is created. If you want to continue past that point, set the fixup flag in the dependency cell to `True`.

In [ ]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from sqlalchemy import text


def find_repo_root(start: Path | None = None) -> Path:
    current = start or Path.cwd()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root from the current working directory")


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

load_dotenv(REPO_ROOT / ".env")

from accent_fleet.config import SQL_DIR, settings
from accent_fleet.db import get_engine, load_sql, run_sql_statement, split_sql_statements, transaction

cfg = settings()
engine = get_engine()

with engine.connect() as conn:
    db_info = conn.execute(
        text(
            """
            SELECT
              current_database() AS database_name,
              current_user AS user_name,
              current_setting('search_path') AS search_path,
              version() AS server_version
            """
        )
    ).mappings().one()

print(f"repo_root: {REPO_ROOT}")
print(f"sql_dir: {SQL_DIR}")
print(f"database: {db_info['database_name']}")
print(f"user: {db_info['user_name']}")
print(f"search_path: {db_info['search_path']}")
print(f"server_version: {db_info['server_version']}")

## 1. Load Environment and Database Connection

This cell loads `.env`, creates the SQLAlchemy engine, and prints the current database identity for traceability.

In [ ]:
SQL_FILE = SQL_DIR / "21_v_device_risk_profile.sql"
SQL_TEXT = SQL_FILE.read_text(encoding="utf-8")

print(f"sql_file: {SQL_FILE}")
print(f"line_count: {len(SQL_TEXT.splitlines())}")
print("\nFirst 60 numbered lines:\n")
for line_number, line in enumerate(SQL_TEXT.splitlines()[:60], start=1):
    print(f"{line_number:>3}: {line}")

## 2. Capture and Display SQL From `21_v_device_risk_profile.sql`

The next cells keep the full file contents in memory and preview the lines that matter for the failure.

In [ ]:
def split_sql_blocks(sql_text: str) -> list[tuple[str, str]]:
    blocks: list[tuple[str, str]] = []
    current_name = "preamble"
    current_lines: list[str] = []

    for raw_line in sql_text.splitlines():
        line = raw_line.rstrip()
        stripped = line.strip()
        if stripped.startswith(("latest_3m AS (", "rolling AS (", "normalized AS (")):
            if current_lines:
                blocks.append((current_name, "\n".join(current_lines).strip()))
            current_name = stripped.split()[0].removesuffix("(")
            current_lines = [line]
        else:
            current_lines.append(line)

    if current_lines:
        blocks.append((current_name, "\n".join(current_lines).strip()))
    return [(name, text) for name, text in blocks if text]


sql_blocks = split_sql_blocks(SQL_TEXT)
print(f"block_count: {len(sql_blocks)}")
for index, (name, block_text) in enumerate(sql_blocks, start=1):
    preview = "\n".join(block_text.splitlines()[:8])
    print(f"\n[{index}] {name}\n{preview}")

## 3. Validate Required Schemas, Tables, and Dependencies

This cell checks the catalog first so the notebook can fail fast when `marts.mart_device_monthly_behavior` is missing.

In [ ]:
STRICT_DEPENDENCY_CHECK = False
CREATE_MART_TABLE_FIXUP = False

required_table = "mart_device_monthly_behavior"
required_columns = [
    "tenant_id",
    "device_id",
    "year_month",
    "total_trips",
    "total_distance_km",
    "overspeed_count",
    "overspeed_severity_high",
    "overspeed_severity_extreme",
    "speed_alert_count",
    "high_speed_trip_ratio",
    "night_trip_ratio",
    "avg_max_speed_kmh",
]

with engine.connect() as conn:
    dependency_report = pd.read_sql_query(
        text(
            """
            SELECT
              c.table_schema,
              c.table_name,
              c.column_name,
              c.data_type
            FROM information_schema.columns c
            WHERE c.table_schema = 'marts'
              AND c.table_name = :table_name
            ORDER BY c.ordinal_position
            """
        ),
        conn,
        params={"table_name": required_table},
    )

schema_exists = not dependency_report.empty
present_columns = set(dependency_report["column_name"].tolist()) if schema_exists else set()
missing_columns = [name for name in required_columns if name not in present_columns]

print(f"marts schema exists: {True}")
print(f"{required_table} exists: {schema_exists}")
print(f"row_count(columns): {len(dependency_report)}")
print(f"missing_columns: {missing_columns}")
if not dependency_report.empty:
    display(dependency_report.head(25))

if STRICT_DEPENDENCY_CHECK:
    assert schema_exists, f"Missing required table: marts.{required_table}"
    assert not missing_columns, f"Missing required columns: {missing_columns}"
else:
    print("Strict dependency checks are disabled. Set STRICT_DEPENDENCY_CHECK = True to raise immediately.")

if CREATE_MART_TABLE_FIXUP and not schema_exists:
    print("Creating marts.mart_device_monthly_behavior from sql/20_mart_device_monthly_behavior.sql ...")
    mart_sql = load_sql("20_mart_device_monthly_behavior.sql")
    mart_statements = split_sql_statements(mart_sql)
    with transaction() as conn:
        for index, statement in enumerate(mart_statements, start=1):
            if statement.lstrip().upper().startswith("INSERT INTO"):
                run_sql_statement(conn, statement, {"touched_months": [], "etl_run_id": -1})
            else:
                run_sql_statement(conn, statement)
    print("mart table creation attempted.")

## 4. Preview Source Data From `marts.mart_device_monthly_behavior`

These checks confirm the mart has the granularity and freshness the view expects before you move to the CTEs.

In [ ]:
with engine.connect() as conn:
    if schema_exists:
        mart_summary = pd.read_sql_query(
            text(
                """
                SELECT
                  COUNT(*) AS row_count,
                  COUNT(DISTINCT tenant_id) AS tenant_count,
                  COUNT(DISTINCT device_id) AS device_count,
                  MIN(year_month) AS min_year_month,
                  MAX(year_month) AS max_year_month
                FROM marts.mart_device_monthly_behavior
                """
            ),
            conn,
        )
        display(mart_summary)

        sample_rows = pd.read_sql_query(
            text(
                """
                SELECT *
                FROM marts.mart_device_monthly_behavior
                ORDER BY year_month DESC, tenant_id, device_id
                LIMIT 10
                """
            ),
            conn,
        )
        display(sample_rows)
    else:
        print("Skipping preview because marts.mart_device_monthly_behavior does not exist yet.")

## 5. Run CTEs Incrementally (`latest_3m`, `rolling`, `normalized`)

These queries mirror the view logic but keep each stage isolated so you can inspect row counts and nulls before combining them.

In [ ]:
def query_frame(sql: str, params: dict[str, object] | None = None) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql_query(text(sql), conn, params=params or {})


if schema_exists:
    latest_3m_sql = """
    WITH latest_3m AS (
      SELECT
        tenant_id,
        device_id,
        year_month,
        ROW_NUMBER() OVER (
          PARTITION BY tenant_id, device_id
          ORDER BY year_month DESC
        ) AS rn
      FROM marts.mart_device_monthly_behavior
    )
    SELECT *
    FROM latest_3m
    ORDER BY tenant_id, device_id, year_month DESC
    LIMIT 20
    """
    latest_3m_df = query_frame(latest_3m_sql)
    print(f"latest_3m shape: {latest_3m_df.shape}")
    display(latest_3m_df)

    rolling_sql = """
    WITH latest_3m AS (
      SELECT
        tenant_id,
        device_id,
        year_month,
        ROW_NUMBER() OVER (
          PARTITION BY tenant_id, device_id
          ORDER BY year_month DESC
        ) AS rn
      FROM marts.mart_device_monthly_behavior
    ),
    rolling AS (
      SELECT
        m.tenant_id,
        m.device_id,
        MAX(m.year_month) AS latest_month,
        SUM(m.total_trips) AS trips_3m,
        SUM(m.total_distance_km) AS distance_3m,
        SUM(m.overspeed_count) AS overspeed_3m,
        SUM(m.overspeed_severity_high + m.overspeed_severity_extreme) AS severe_overspeed_3m,
        SUM(m.speed_alert_count) AS alerts_3m,
        AVG(m.high_speed_trip_ratio) AS high_speed_trip_ratio_3m,
        AVG(m.night_trip_ratio) AS night_trip_ratio_3m,
        MAX(m.avg_max_speed_kmh) AS max_recorded_speed_3m
      FROM marts.mart_device_monthly_behavior m
      JOIN latest_3m l USING (tenant_id, device_id, year_month)
      WHERE l.rn <= 3
      GROUP BY m.tenant_id, m.device_id
    )
    SELECT *
    FROM rolling
    ORDER BY latest_month DESC, tenant_id, device_id
    LIMIT 20
    """
    rolling_df = query_frame(rolling_sql)
    print(f"rolling shape: {rolling_df.shape}")
    display(rolling_df)

    normalized_sql = """
    WITH latest_3m AS (
      SELECT
        tenant_id,
        device_id,
        year_month,
        ROW_NUMBER() OVER (
          PARTITION BY tenant_id, device_id
          ORDER BY year_month DESC
        ) AS rn
      FROM marts.mart_device_monthly_behavior
    ),
    rolling AS (
      SELECT
        m.tenant_id,
        m.device_id,
        MAX(m.year_month) AS latest_month,
        SUM(m.total_trips) AS trips_3m,
        SUM(m.total_distance_km) AS distance_3m,
        SUM(m.overspeed_count) AS overspeed_3m,
        SUM(m.overspeed_severity_high + m.overspeed_severity_extreme) AS severe_overspeed_3m,
        SUM(m.speed_alert_count) AS alerts_3m,
        AVG(m.high_speed_trip_ratio) AS high_speed_trip_ratio_3m,
        AVG(m.night_trip_ratio) AS night_trip_ratio_3m,
        MAX(m.avg_max_speed_kmh) AS max_recorded_speed_3m
      FROM marts.mart_device_monthly_behavior m
      JOIN latest_3m l USING (tenant_id, device_id, year_month)
      WHERE l.rn <= 3
      GROUP BY m.tenant_id, m.device_id
    ),
    normalized AS (
      SELECT
        *,
        LEAST(1.0, (COALESCE(overspeed_3m, 0) / NULLIF(distance_3m, 0) * 100) / 10.0) AS n_overspeed_rate,
        CASE WHEN overspeed_3m > 0 THEN severe_overspeed_3m::DOUBLE PRECISION / overspeed_3m ELSE 0 END AS n_severe_share,
        LEAST(1.0, COALESCE(high_speed_trip_ratio_3m, 0) / 0.3) AS n_high_speed_ratio,
        LEAST(1.0, (COALESCE(alerts_3m, 0) / NULLIF(distance_3m, 0) * 100) / 20.0) AS n_alert_rate,
        LEAST(1.0, COALESCE(night_trip_ratio_3m, 0) / 0.3) AS n_night,
        LEAST(1.0, COALESCE(max_recorded_speed_3m, 0) / 200.0) AS n_max_speed
      FROM rolling
      WHERE trips_3m >= 10
    )
    SELECT *
    FROM normalized
    ORDER BY latest_month DESC, tenant_id, device_id
    LIMIT 20
    """
    normalized_df = query_frame(normalized_sql)
    print(f"normalized shape: {normalized_df.shape}")
    print("normalized null counts:")
    print(normalized_df.isna().sum().sort_values(ascending=False).head(15))
    display(normalized_df)
else:
    print("Skipping CTE runs because the mart table is missing.")

## 6. Create/Replace View Safely With Guard Clauses

This cell mirrors Prefect's `task_ensure_views()` behavior, but catches and prints database errors so you can see the exact failure before deciding whether to repair the dependency chain.

In [ ]:
VIEW_SQL = SQL_TEXT


def try_create_view(sql_text: str) -> None:
    try:
        with engine.begin() as conn:
            conn.execute(text(sql_text))
        print("marts.v_device_risk_profile created or replaced successfully.")
    except Exception as exc:
        print(f"view creation failed: {type(exc).__name__}: {exc}")


if schema_exists:
    try_create_view(VIEW_SQL)
else:
    print("Skipping view creation because the mart table is missing.")

## 7. Validate Output Columns, Row Counts, and Risk Category Distribution

Run this after the view exists to confirm the final contract and distribution shape.

In [ ]:
if schema_exists:
    with engine.connect() as conn:
        try:
            view_frame = pd.read_sql_query(
                text(
                    """
                    SELECT *
                    FROM marts.v_device_risk_profile
                    ORDER BY latest_month DESC, tenant_id, device_id
                    LIMIT 20
                    """
                ),
                conn,
            )
            print(f"view shape: {view_frame.shape}")
            display(view_frame)

            view_counts = pd.read_sql_query(
                text(
                    """
                    SELECT
                      COUNT(*) AS row_count,
                      COUNT(DISTINCT (tenant_id, device_id)) AS unique_device_keys,
                      MIN(risk_score) AS min_risk_score,
                      MAX(risk_score) AS max_risk_score
                    FROM marts.v_device_risk_profile
                    """
                ),
                conn,
            )
            display(view_counts)

            category_distribution = pd.read_sql_query(
                text(
                    """
                    SELECT risk_category, COUNT(*) AS row_count
                    FROM marts.v_device_risk_profile
                    GROUP BY risk_category
                    ORDER BY row_count DESC, risk_category
                    """
                ),
                conn,
            )
            display(category_distribution)
        except Exception as exc:
            print(f"view validation failed: {type(exc).__name__}: {exc}")
else:
    print("Skipping validation because the view has not been created yet.")

## 8. Re-runnable Debug Helpers for Prefect Parity

These helpers match the repository's execution style closely enough that you can compare notebook output with Prefect task output line by line.

In [ ]:
def timed_exec(label: str, fn, *args, **kwargs):
    started_at = time.perf_counter()
    try:
        result = fn(*args, **kwargs)
        elapsed = time.perf_counter() - started_at
        print(f"{label}: ok in {elapsed:.3f}s")
        return result
    except Exception as exc:
        elapsed = time.perf_counter() - started_at
        print(f"{label}: failed in {elapsed:.3f}s -> {type(exc).__name__}: {exc}")
        raise


def run_sql_statement_prefect_like(sql_text: str, params: dict[str, object] | None = None) -> None:
    with engine.begin() as conn:
        timed_exec("run_sql_statement", conn.execute, text(sql_text), params or {})


def run_sql_file_prefect_like(file_name: str, params: dict[str, object] | None = None) -> None:
    file_path = SQL_DIR / file_name
    sql_text = file_path.read_text(encoding="utf-8")
    statements = split_sql_statements(sql_text)
    print(f"{file_name}: {len(statements)} statement(s)")
    with engine.begin() as conn:
        for index, statement in enumerate(statements, start=1):
            label = f"{file_name} [{index}/{len(statements)}]"
            if statement.lstrip().upper().startswith("INSERT INTO") and params:
                timed_exec(label, conn.execute, text(statement), params)
            else:
                timed_exec(label, conn.execute, text(statement), {})


print("Helpers ready: timed_exec, run_sql_statement_prefect_like, run_sql_file_prefect_like")

## 9. Known Fixes to Try Next

1. Create `marts.mart_device_monthly_behavior` before `task_ensure_views()` runs, or move the view task until after the mart is materialized.
2. Leave `STRICT_DEPENDENCY_CHECK` on when you want the notebook to stop immediately on missing tables.
3. Turn `CREATE_MART_TABLE_FIXUP` on only when you want the notebook to attempt the missing-table repair automatically.
4. If you want the bootstrap flow itself fixed next, the same ordering issue should be handled in `src/accent_fleet/pipeline/flow_batch.py`.